In [47]:
import polars as pl

# Clientes

In [48]:
clientes = pl.read_csv("../data/clientes.csv")

clientes.head()

id_cliente,nombre,apellidos,sexo,edad,ciudad,provincia,codigo_postal,fecha_alta,socio,tipo_cliente
i64,str,str,str,i64,str,str,i64,str,bool,str
1,"""Álvaro""","""Castro Navarro""","""Mujer""",24,"""Valencia""","""Valencia""",35314,"""2023-04-25""",false,"""Normal"""
2,"""Miguel""","""Martín Sánchez""","""Mujer""",40,"""Santander""","""Cantabria""",49225,"""2022-01-22""",false,"""Ocasional"""
3,"""Elena""","""Romero Ortega""","""Mujer""",61,"""Alicante""","""Alicante""",23449,"""2023-11-14""",true,"""Normal"""
4,"""María""","""Romero Pérez""","""Hombre""",49,"""Salamanca""","""Salamanca""",18478,"""2024-06-24""",true,"""Normal"""
5,"""Alejandro""","""Jiménez Castro""","""Mujer""",28,"""Logroño""","""La Rioja""",36179,"""2022-11-16""",true,"""Frecuente"""


In [49]:
# Convertir la fecha a tipo Date
clientes = clientes.with_columns(
    pl.col("fecha_alta").str.to_date()
)

clientes.dtypes

[Int64,
 String,
 String,
 String,
 Int64,
 String,
 String,
 Int64,
 Date,
 Boolean,
 String]

In [50]:
# Creamos el año de alta
clientes = clientes.with_columns(
    pl.col("fecha_alta")
      .dt.year()
      .alias("anio_alta")
)

clientes.select(
    "fecha_alta",
    "anio_alta"
).head()

fecha_alta,anio_alta
date,i32
2023-04-25,2023
2022-01-22,2022
2023-11-14,2023
2024-06-24,2024
2022-11-16,2022


In [51]:
# Creamos el mes de alta
clientes = clientes.with_columns(
    pl.col("fecha_alta")
      .dt.month()
      .alias("mes_alta")
)

In [52]:
# Creamos el día de la semana
clientes = clientes.with_columns(
    pl.col("fecha_alta")
      .dt.weekday()
      .alias("dia_semana")
)

In [53]:
# Creamos diferentes grupos de edad
clientes = clientes.with_columns(

    pl.when(pl.col("edad") < 30)
      .then(pl.lit("Joven"))

    .when(pl.col("edad") < 50)
      .then(pl.lit("Adulto"))

    .when(pl.col("edad") < 65)
      .then(pl.lit("Senior"))

    .otherwise(pl.lit("Veterano"))

    .alias("grupo_edad")

)

In [54]:
# Creamos una nueva columna con la antigüedad
from datetime import date

clientes = clientes.with_columns(

    (
        date.today().year -
        pl.col("fecha_alta").dt.year()
    ).alias("antiguedad")

)

In [55]:
# Clasificamos a los clientes por antigüedad
clientes = clientes.with_columns(

    pl.when(pl.col("antiguedad") < 2)
      .then(pl.lit("Nuevo"))

    .when(pl.col("antiguedad") < 4)
      .then(pl.lit("Habitual"))

    .otherwise(pl.lit("Veterano"))

    .alias("nivel_cliente")

)

In [56]:
# Reordenamos las columnas
clientes = clientes.select(

    "id_cliente",

    "nombre",

    "apellidos",

    "sexo",

    "edad",

    "grupo_edad",

    "ciudad",

    "provincia",

    "codigo_postal",

    "fecha_alta",

    "anio_alta",

    "mes_alta",

    "dia_semana",

    "antiguedad",

    "nivel_cliente",

    "socio",

    "tipo_cliente"

)

clientes.head()

id_cliente,nombre,apellidos,sexo,edad,grupo_edad,ciudad,provincia,codigo_postal,fecha_alta,anio_alta,mes_alta,dia_semana,antiguedad,nivel_cliente,socio,tipo_cliente
i64,str,str,str,i64,str,str,str,i64,date,i32,i8,i8,i32,str,bool,str
1,"""Álvaro""","""Castro Navarro""","""Mujer""",24,"""Joven""","""Valencia""","""Valencia""",35314,2023-04-25,2023,4,2,3,"""Habitual""",false,"""Normal"""
2,"""Miguel""","""Martín Sánchez""","""Mujer""",40,"""Adulto""","""Santander""","""Cantabria""",49225,2022-01-22,2022,1,6,4,"""Veterano""",false,"""Ocasional"""
3,"""Elena""","""Romero Ortega""","""Mujer""",61,"""Senior""","""Alicante""","""Alicante""",23449,2023-11-14,2023,11,2,3,"""Habitual""",true,"""Normal"""
4,"""María""","""Romero Pérez""","""Hombre""",49,"""Adulto""","""Salamanca""","""Salamanca""",18478,2024-06-24,2024,6,1,2,"""Habitual""",true,"""Normal"""
5,"""Alejandro""","""Jiménez Castro""","""Mujer""",28,"""Joven""","""Logroño""","""La Rioja""",36179,2022-11-16,2022,11,3,4,"""Veterano""",true,"""Frecuente"""


# Productos

In [57]:
productos = pl.read_csv("../data/productos.csv")

productos.head()

id_producto,nombre,categoria,marca,precio,coste
i64,str,str,str,f64,f64
1,"""Barra""","""Panadería""","""Casa Tarradellas""",8.49,4.38
2,"""Snacks perro""","""Mascotas""","""Casa Tarradellas""",9.92,6.73
3,"""Detergente""","""Limpieza""","""Pascual""",5.45,2.48
4,"""Calamares""","""Pescados""","""Font Vella""",14.94,9.11
5,"""Turrón""","""Dulces""","""Font Vella""",4.37,2.18


In [58]:
productos.dtypes

[Int64, String, String, String, Float64, Float64]

In [59]:
productos = productos.with_columns(
    (pl.col("precio") - pl.col("coste")).alias("beneficio")
)

productos.select(
    "nombre",
    "precio",
    "coste",
    "beneficio"
).head()


nombre,precio,coste,beneficio
str,f64,f64,f64
"""Barra""",8.49,4.38,4.11
"""Snacks perro""",9.92,6.73,3.19
"""Detergente""",5.45,2.48,2.97
"""Calamares""",14.94,9.11,5.83
"""Turrón""",4.37,2.18,2.19


In [60]:
# Creamos diferentes grupos de beneficio
productos = productos.with_columns(
    pl.when(pl.col("beneficio") < 2)
      .then(pl.lit("Poco rentable"))
      .when(
          (pl.col("beneficio") >= 2) &
          (pl.col("beneficio") < 5)
      )
      .then(pl.lit("Rentable"))
      .otherwise(pl.lit("Muy rentable"))
      .alias("grupo_beneficio")
)

productos.head()

id_producto,nombre,categoria,marca,precio,coste,beneficio,grupo_beneficio
i64,str,str,str,f64,f64,f64,str
1,"""Barra""","""Panadería""","""Casa Tarradellas""",8.49,4.38,4.11,"""Rentable"""
2,"""Snacks perro""","""Mascotas""","""Casa Tarradellas""",9.92,6.73,3.19,"""Rentable"""
3,"""Detergente""","""Limpieza""","""Pascual""",5.45,2.48,2.97,"""Rentable"""
4,"""Calamares""","""Pescados""","""Font Vella""",14.94,9.11,5.83,"""Muy rentable"""
5,"""Turrón""","""Dulces""","""Font Vella""",4.37,2.18,2.19,"""Rentable"""


# Ventas

In [61]:
ventas = pl.read_csv("../data/ventas.csv")

ventas.head()

id_venta,fecha,id_cliente,id_producto,cantidad,metodo_pago,descuento
i64,str,i64,i64,i64,str,i64
1,"""2025-04-07 09:00:00""",276,1,3,"""Tarjeta""",10
2,"""2025-03-12 09:00:00""",34,12,3,"""Tarjeta""",5
3,"""2025-08-24 09:00:00""",27,134,3,"""Tarjeta""",0
4,"""2025-08-20 16:00:00""",338,63,1,"""Bizum""",0
5,"""2025-02-03 21:00:00""",206,68,4,"""Cheque regalo""",0


In [62]:
# Convertir la fecha a tipo Date
ventas = ventas.with_columns(
    pl.col("fecha")
      .str.to_datetime(format="%Y-%m-%d %H:%M:%S")
      .dt.date()
      .alias("fecha")
)

ventas.dtypes

[Int64, Date, Int64, Int64, Int64, String, Int64]

In [63]:
# Creamos el año de alta
ventas = ventas.with_columns(
    pl.col("fecha")
      .dt.month()
      .alias("mes_compra")
)

ventas.select(
    "fecha",
    "mes_compra"
).head()

fecha,mes_compra
date,i8
2025-04-07,4
2025-03-12,3
2025-08-24,8
2025-08-20,8
2025-02-03,2


In [64]:
# Creamos el año de alta
ventas = ventas.with_columns(
    pl.col("fecha")
      .dt.weekday()
      .alias("dia_semana")
)

ventas.select(
    "fecha",
    "dia_semana"
).head()


fecha,dia_semana
date,i8
2025-04-07,1
2025-03-12,3
2025-08-24,7
2025-08-20,3
2025-02-03,1


In [65]:
# Creamos diferentes grupos de beneficio
ventas = ventas.with_columns(
    pl.when(pl.col("metodo_pago") == "Tarjeta")
      .then(pl.lit("CaixaBank"))
      .when((pl.col("metodo_pago") == "Bizum"))
      .then(pl.lit("Santander"))
      .otherwise(pl.lit("En caja"))
      .alias("grupo_bancos")
)

ventas.head()

id_venta,fecha,id_cliente,id_producto,cantidad,metodo_pago,descuento,mes_compra,dia_semana,grupo_bancos
i64,date,i64,i64,i64,str,i64,i8,i8,str
1,2025-04-07,276,1,3,"""Tarjeta""",10,4,1,"""CaixaBank"""
2,2025-03-12,34,12,3,"""Tarjeta""",5,3,3,"""CaixaBank"""
3,2025-08-24,27,134,3,"""Tarjeta""",0,8,7,"""CaixaBank"""
4,2025-08-20,338,63,1,"""Bizum""",0,8,3,"""Santander"""
5,2025-02-03,206,68,4,"""Cheque regalo""",0,2,1,"""En caja"""


In [66]:
# Creamos diferentes grupos de beneficio
ventas = ventas.with_columns(
    pl.when(pl.col("dia_semana") >5 )
      .then(pl.lit("Find_de_semana"))
      .otherwise(pl.lit("Entre semana"))
      .alias("temporalidad_semana")
)

ventas.head()

id_venta,fecha,id_cliente,id_producto,cantidad,metodo_pago,descuento,mes_compra,dia_semana,grupo_bancos,temporalidad_semana
i64,date,i64,i64,i64,str,i64,i8,i8,str,str
1,2025-04-07,276,1,3,"""Tarjeta""",10,4,1,"""CaixaBank""","""Entre semana"""
2,2025-03-12,34,12,3,"""Tarjeta""",5,3,3,"""CaixaBank""","""Entre semana"""
3,2025-08-24,27,134,3,"""Tarjeta""",0,8,7,"""CaixaBank""","""Find_de_semana"""
4,2025-08-20,338,63,1,"""Bizum""",0,8,3,"""Santander""","""Entre semana"""
5,2025-02-03,206,68,4,"""Cheque regalo""",0,2,1,"""En caja""","""Entre semana"""


**Ejercicio**

Hacer lo mismo para productos y ventas.